# Experiment 4: Logistic Regression Hyperparameter Tuning

Grid search optimization for Logistic Regression hyperparameters (solvers, penalties, C regularization, l1_ratio) on TF-IDF features.

In [ ]:
import os
import re
import string
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import mlflow
import mlflow.sklearn
import dagshub

warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore")

## 1. Load and Vectorize Data

In [ ]:
DATA_PATH = '../data/balanced_sentiment_dataset.csv' if os.path.exists('../data/balanced_sentiment_dataset.csv') else 'data/balanced_sentiment_dataset.csv'

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

df = pd.read_csv(DATA_PATH)
df["text"] = df["text"].astype(str).apply(preprocess_text)

vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df["text"])
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training shape: {X_train.shape}, Test shape: {X_test.shape}")

## 2. GridSearchCV for Logistic Regression

In [ ]:
param_grid = [
    {
        "solver": ["liblinear"],
        "penalty": ["l1", "l2"],
        "C": [0.1, 1.0, 10.0]
    },
    {
        "solver": ["saga"],
        "penalty": ["elasticnet"],
        "C": [0.1, 1.0, 10.0],
        "l1_ratio": [0.2, 0.5, 0.8]
    },
    {
        "solver": ["lbfgs"],
        "penalty": ["l2"],
        "C": [0.1, 1.0, 10.0]
    }
]

grid_search = GridSearchCV(
    LogisticRegression(max_iter=500),
    param_grid=param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation F1-Score: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print(f"Holdout Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Holdout Test F1-Score: {f1_score(y_test, y_pred):.4f}")